In [1]:
from pathlib import Path
import importlib.util
import json
import site
import sys
import warnings

ROOT = Path.cwd()
if not (ROOT / "Data").exists() and (ROOT.parent / "Data").exists():
    ROOT = ROOT.parent

candidate_sites = []
user_site = site.getusersitepackages()
if user_site:
    candidate_sites.append(Path(user_site))
candidate_sites.append(Path.home() / 'AppData' / 'Roaming' / 'Python' / f'Python{sys.version_info.major}{sys.version_info.minor}' / 'site-packages')

for candidate in candidate_sites:
    candidate_text = str(candidate)
    if candidate_text not in sys.path:
        sys.path.insert(0, candidate_text)

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

DATA_DIR = ROOT / 'Data' / 'DementiaData'
MODELS_DIR = ROOT / 'Models'
SEED = 42

spec = importlib.util.spec_from_file_location('real_time_detection', ROOT / 'Real_Time_Detection' / 'real_time_detection.py')
rtd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rtd)

ROOT, DATA_DIR, MODELS_DIR


(WindowsPath('c:/Users/vipra/OneDrive/Documents/GitHub/Audio-Based-Neurological-Disease-Detection-Project'),
 WindowsPath('c:/Users/vipra/OneDrive/Documents/GitHub/Audio-Based-Neurological-Disease-Detection-Project/Data/DementiaData'),
 WindowsPath('c:/Users/vipra/OneDrive/Documents/GitHub/Audio-Based-Neurological-Disease-Detection-Project/Models'))

In [2]:
folder_to_label = {
    'Dementia': 'dementia',
    'Dysarthria': 'dysarthria',
    'Parkinsons': 'parkinsons',
    'NoDementia': 'healthy',
    'Female_Non_Dysarthria': 'healthy',
    'Male_Non_Dysarthria': 'healthy',
}

def speaker_id_for_file(folder_name, wav_path):
    rel = wav_path.relative_to(DATA_DIR / folder_name)
    parts = list(rel.parts)
    if folder_name == 'Dysarthria':
        if len(parts) >= 2:
            return folder_name + '_' + parts[0] + '_' + parts[1]
        return folder_name + '_' + wav_path.parent.name
    if folder_name in {'Female_Non_Dysarthria', 'Male_Non_Dysarthria'}:
        if parts:
            return folder_name + '_' + parts[0]
        return folder_name + '_' + wav_path.parent.name
    if len(parts) >= 2:
        return folder_name + '_' + parts[0] + '_' + parts[1]
    return folder_name + '_' + wav_path.parent.name

rows = []

for folder_name, label in folder_to_label.items():
    folder_path = DATA_DIR / folder_name
    if not folder_path.exists():
        print('missing folder:', folder_path)
        continue

    wav_files = sorted(folder_path.rglob('*.wav'))
    print(folder_name, len(wav_files))

    for wav_path in wav_files:
        try:
            audio = rtd.load_audio(wav_path)
            audio_features = rtd.extract_features(audio).astype(np.float32)
            transcript_text = rtd.find_matching_transcript(wav_path)
            dementia_text_score = rtd.dementia_text_audio_probability(wav_path, audio_features, transcript_text)
            rows.append({
                'file': str(wav_path),
                'folder_name': folder_name,
                'label': label,
                'speaker_id': speaker_id_for_file(folder_name, wav_path),
                'used_transcript': bool(transcript_text),
                'dementia_text_score': float(dementia_text_score) if dementia_text_score is not None else np.nan,
                'audio_features': audio_features,
            })
        except Exception as e:
            print('skipped', wav_path, e)

clip_df = pd.DataFrame(rows)
print('clip rows:', len(clip_df))
print(clip_df['label'].value_counts())
print('unique speakers:', clip_df['speaker_id'].nunique())
clip_df.head()


Dementia 131
Dysarthria 1363
skipped c:\Users\vipra\OneDrive\Documents\GitHub\Audio-Based-Neurological-Disease-Detection-Project\Data\DementiaData\Dysarthria\Female\F01\Session1\Wav\0067.wav 
skipped c:\Users\vipra\OneDrive\Documents\GitHub\Audio-Based-Neurological-Disease-Detection-Project\Data\DementiaData\Dysarthria\Female\F01\Session1\Wav\0068.wav 
Parkinsons 527
NoDementia 104
Female_Non_Dysarthria 776
Male_Non_Dysarthria 2177
clip rows: 5076
label
healthy       3057
dysarthria    1361
parkinsons     527
dementia       131
Name: count, dtype: int64
unique speakers: 257


,file,folder_name,label,speaker_id,used_transcript,dementia_text_score,audio_features
0,c:\Users\vipra\OneDrive\Documents\GitHub\Audio...,Dementia,dementia,Dementia_Abe Burrows_AbeBurrows_5.wav,True,0.938370,"[71.01931, 0.05159282, 0.088297494, 1.0, 0.234..."
1,c:\Users\vipra\OneDrive\Documents\GitHub\Audio...,Dementia,dementia,Dementia_Aileen Hernandez_aileenhernandez_0.wav,True,0.978025,"[46.86519, 0.105095334, 0.17251047, 1.0, 0.555..."
2,c:\Users\vipra\OneDrive\Documents\GitHub\Audio...,Dementia,dementia,Dementia_Aileen Hernandez_aileenhernandez_5_1.wav,True,0.339630,"[37.8835, 0.08994625, 0.14905396, 1.0, 0.28024..."
3,c:\Users\vipra\OneDrive\Documents\GitHub\Audio...,Dementia,dementia,Dementia_Aileen Hernandez_aileenhernandez_5_2.wav,True,0.982503,"[89.61431, 0.043825112, 0.08007261, 1.0, 0.373..."
4,c:\Users\vipra\OneDrive\Documents\GitHub\Audio...,Dementia,dementia,Dementia_Alan Ramsey_alanramsey_10.wav,True,0.501334,"[54.086063, 0.043046553, 0.08071019, 1.0, 0.18..."


In [3]:
speaker_rows = []

for speaker_id, group in clip_df.groupby('speaker_id'):
    feature_matrix = np.vstack(group['audio_features'].to_list())
    mean_features = feature_matrix.mean(axis=0)
    std_features = feature_matrix.std(axis=0)

    text_scores = group['dementia_text_score'].dropna().to_numpy(dtype=np.float32)
    if text_scores.size == 0:
        text_summary = np.asarray([0.0, 0.0, 0.0], dtype=np.float32)
    else:
        text_summary = np.asarray([
            float(text_scores.mean()),
            float(text_scores.max()),
            float(text_scores.std()),
        ], dtype=np.float32)

    meta_features = np.asarray([
        float(len(group)),
        float(group['used_transcript'].mean()),
    ], dtype=np.float32)

    final_features = np.concatenate([mean_features, std_features, text_summary, meta_features]).astype(np.float32)

    speaker_rows.append({
        'speaker_id': speaker_id,
        'label': group['label'].iloc[0],
        'folder_name': group['folder_name'].iloc[0],
        'clip_count': int(len(group)),
        'features': final_features,
    })

speaker_df = pd.DataFrame(speaker_rows)
print('speaker rows:', len(speaker_df))
print(speaker_df['label'].value_counts())
speaker_df.head()


speaker rows: 257
label
dementia      131
healthy       110
parkinsons     11
dysarthria      5
Name: count, dtype: int64


,speaker_id,label,folder_name,clip_count,features
0,Dementia_Abe Burrows_AbeBurrows_5.wav,dementia,Dementia,1,"[71.01931, 0.05159282, 0.088297494, 1.0, 0.234..."
1,Dementia_Aileen Hernandez_aileenhernandez_0.wav,dementia,Dementia,1,"[46.86519, 0.105095334, 0.17251047, 1.0, 0.555..."
2,Dementia_Aileen Hernandez_aileenhernandez_5_1.wav,dementia,Dementia,1,"[37.8835, 0.08994625, 0.14905396, 1.0, 0.28024..."
3,Dementia_Aileen Hernandez_aileenhernandez_5_2.wav,dementia,Dementia,1,"[89.61431, 0.043825112, 0.08007261, 1.0, 0.373..."
4,Dementia_Alan Ramsey_alanramsey_10.wav,dementia,Dementia,1,"[54.086063, 0.043046553, 0.08071019, 1.0, 0.18..."


In [4]:
X = np.vstack(speaker_df['features'].to_list())
y = speaker_df['label'].to_numpy()

models = {
    'extra_trees': ExtraTreesClassifier(
        n_estimators=400,
        random_state=SEED,
        class_weight='balanced',
        n_jobs=-1,
        min_samples_leaf=1,
    ),
    'random_forest': RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        class_weight='balanced_subsample',
        n_jobs=-1,
        min_samples_leaf=1,
    ),
    'logistic_regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=4000, class_weight='balanced', random_state=SEED)),
    ]),
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED)
summary_rows = []

for model_name, model in models.items():
    fold_accuracies = []
    fold_balanced = []

    for train_idx, valid_idx in cv.split(X, y):
        model.fit(X[train_idx], y[train_idx])
        pred = model.predict(X[valid_idx])
        fold_accuracies.append(accuracy_score(y[valid_idx], pred))
        fold_balanced.append(balanced_accuracy_score(y[valid_idx], pred))

    summary_rows.append({
        'model': model_name,
        'cv_accuracy_mean': round(float(np.mean(fold_accuracies)), 4),
        'cv_accuracy_std': round(float(np.std(fold_accuracies)), 4),
        'cv_balanced_accuracy_mean': round(float(np.mean(fold_balanced)), 4),
        'cv_balanced_accuracy_std': round(float(np.std(fold_balanced)), 4),
    })

results_df = pd.DataFrame(summary_rows).sort_values(
    ['cv_balanced_accuracy_mean', 'cv_accuracy_mean'],
    ascending=False,
).reset_index(drop=True)
results_df


,model,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_balanced_accuracy_std
0,random_forest,0.8013,0.0560,0.8878,0.0300
1,extra_trees,0.8227,0.0582,0.8766,0.0959
2,logistic_regression,0.7078,0.0876,0.7941,0.1080


In [5]:
best_model_name = results_df.iloc[0]['model']
best_model = models[best_model_name]

X_train, X_test, y_train, y_test, speaker_train, speaker_test = train_test_split(
    X,
    y,
    speaker_df['speaker_id'].to_numpy(),
    test_size=0.25,
    random_state=SEED,
    stratify=y,
)

best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)

summary = {
    'best_model': best_model_name,
    'test_accuracy': round(float(accuracy_score(y_test, best_pred)), 4),
    'test_balanced_accuracy': round(float(balanced_accuracy_score(y_test, best_pred)), 4),
    'train_speakers': int(len(y_train)),
    'test_speakers': int(len(y_test)),
    'feature_count': int(X.shape[1]),
}
print(json.dumps(summary, indent=2))

package = {
    'model': best_model,
    'classes': sorted(speaker_df['label'].unique().tolist()),
    'feature_count': int(X.shape[1]),
    'description': 'Speaker-level multiclass detector using aggregated audio features and dementia text hints',
}
joblib.dump(package, MODELS_DIR / 'multiclass_detector.joblib')
print('saved:', MODELS_DIR / 'multiclass_detector.joblib')


{
  "best_model": "random_forest",
  "test_accuracy": 0.8308,
  "test_balanced_accuracy": 0.9058,
  "train_speakers": 192,
  "test_speakers": 65,
  "feature_count": 2103
}
saved: c:\Users\vipra\OneDrive\Documents\GitHub\Audio-Based-Neurological-Disease-Detection-Project\Models\multiclass_detector.joblib


In [6]:
print(classification_report(y_test, best_pred, digits=4))

labels = ['dementia', 'dysarthria', 'healthy', 'parkinsons']
cm = confusion_matrix(y_test, best_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df


              precision    recall  f1-score   support

    dementia     0.8108    0.9091    0.8571        33
  dysarthria     0.5000    1.0000    0.6667         1
     healthy     0.8696    0.7143    0.7843        28
  parkinsons     1.0000    1.0000    1.0000         3

    accuracy                         0.8308        65
   macro avg     0.7951    0.9058    0.8270        65
weighted avg     0.8401    0.8308    0.8294        65



,dementia,dysarthria,healthy,parkinsons
dementia,30,0,3,0
dysarthria,0,1,0,0
healthy,7,1,20,0
parkinsons,0,0,0,3
